# Ο δακτύλιος: τα δύο ανταγωνιστικά κελιά με το σωστό prompt

**Το ανοιχτό σημείο.** Μέχρι τις 20/9 το cheap-talk system prompt περιέγραφε
**αστέρα σε κάθε τοπολογία**. Αυτό αφορά 500 runs του δακτυλίου. Το ablation
ξανατρέχει τα κελιά του session A στο PD και δεν βρήκε καμία μετακίνηση
(19 από 20 κελιά αμετάβλητα), αλλά τα sessions **B και C** του δακτυλίου δεν
ξανατρέχτηκαν ποτέ. Για αυτά, η διατριβή στηρίζεται σε **επέκταση** του
αποτελέσματος του ablation, όχι σε μέτρηση.

**Τι κλείνει αυτό το notebook.** Τα δύο κελιά που κουβαλούν τον πυρήνα του
RQ3 και ολόκληρο το RQ4:

| κελί | πού ζει το frame | τι στηρίζει |
|---|---|---|
| `framing_competitive` | στα **μηνύματα** | η μοναδική συστηματική οδός βλάβης |
| `framing_competitive_context[cheap_talk]` | στο **system prompt**, ανοιχτό κανάλι | «το ανοιχτό κανάλι υπερισχύει του frame» |

Αν αυτά τα δύο δεν κουνηθούν με το διορθωμένο prompt, το επιχείρημα για τα
υπόλοιπα κελιά B/C γίνεται πολύ ισχυρότερο, με το ένα τέταρτο του κόστους
μιας πλήρους επανάληψης του PD.

**Τι μένει εκτός και γιατί.** Το `no_comm` σκέλος του context framing δεν
είχε ποτέ πρόταση δρομολόγησης, οπότε είναι ήδη σωστό· τα υπόλοιπα framings
(business, team) και όλο το SH μένουν με το παλιό prompt και δηλώνονται στο
κείμενο.

## Τι δεν αλλάζει

Ίδιο παιχνίδι και αποδόσεις, 16 γύροι, κρυφός ορίζοντας, μνήμη 10, T=0,7,
μηνύματα ως 20 λέξεις, n=5, PD. Τα `max_tokens` τα δίνει το `campaign.py` ανά
μοντέλο και session, **τα ίδια που είχαν τα παλιά runs** αυτών των κελιών:
session B για το `framing_competitive`, session A για το context framing.

Αλλάζει **μία πρόταση** στο cheap-talk system prompt: «each agent sends a
single message that reaches the two agents adjacent to it in the ring»
αντί για την περιγραφή αστέρα. Τα runs πάνε σε δικά τους δέντρα, δίπλα στο
ablation.

## Setup

1. Settings → Accelerator → **GPU T4 x2**
2. Settings → Internet → **On**
3. Add-ons → Secrets → `HF_TOKEN` (για Llama και τα δύο gemma)

Μετά **Save & Run All**, αλλάζοντας μόνο το `STEP`.

| βήμα | μοντέλα | runs | ~ώρες |
|---|---|---|---|
| 1 | gemma-2-2b, Qwen2.5-7B | 20 | 3,3 |
| 2 | Llama-3.1-8B, Qwen3-4B | 20 | 4,8 |
| 3 | gemma-2-9b | 10 | 3,5 |

**Σύνολο 50 runs, ~11,6 ώρες.**

In [ ]:
!pip install -q bitsandbytes python-dotenv

import torch
assert torch.cuda.is_available(), 'GPU off -- Settings -> Accelerator -> GPU T4 x2'
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
GITHUB_REPO = 'https://github.com/stsimpe/cheaptalk_bench.git'
REPO_DIR = '/kaggle/working/repo'

import os, subprocess
if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', GITHUB_REPO, REPO_DIR], check=True)
os.chdir(REPO_DIR)

assert os.path.exists('campaign.py'), f'campaign.py δεν είναι στο {os.getcwd()}'
for f in ('campaign.py', 'run_all_scenarios.py'):
    assert '--topology-aware-comm-prompt' in open(f, encoding='utf-8').read(), (
        f'Το {f} δεν έχει --topology-aware-comm-prompt -- κάνε git push πρώτα.')

print('HEAD:', subprocess.run(['git', 'log', '--oneline', '-1'],
                              capture_output=True, text=True).stdout.strip())


## Ο έλεγχος που κρίνει το πείραμα

Χτίζει το cheap-talk prompt του δακτυλίου με και χωρίς τη διόρθωση, και
επιβεβαιώνει ότι αλλάζει **ακριβώς μία γραμμή** και ότι δεν μένει κείμενο
αστέρα. Είναι ο ίδιος έλεγχος που πέρασε το ablation.

In [ ]:
import sys
sys.path.insert(0, '.')
from games import GAMES
from topology import make_topology
from prompts import build_system_prompt
from message_policies import CONTEXT_FRAMING_PARAGRAPHS

topo = make_topology('cycle', 4)
assert topo.neighbors(0) == [1, 3], topo.neighbors(0)

for ctx_name in ('none', 'competitive'):
    ctx = CONTEXT_FRAMING_PARAGRAPHS[ctx_name]
    common = dict(game=GAMES['pd'], condition='cheap_talk', n_neighbors=2,
                  total_agents=4, topology_text=topo.describe(0),
                  context_framing_text=ctx)
    old = build_system_prompt(**common)
    new = build_system_prompt(**common,
                              communication_text=topo.describe_communication(0))
    diff = [l for l in new.splitlines() if l not in old.splitlines()]
    assert len(diff) == 1, f'{ctx_name}: άλλαξαν {len(diff)} γραμμές -- σταμάτα'
    assert 'central agent' in old and 'central agent' not in new, ctx_name
    assert 'adjacent to it in the ring' in new, ctx_name
    if ctx:
        assert ctx in new, f'{ctx_name}: χάθηκε η παράγραφος context -- σταμάτα'
    print(f'{ctx_name:12s} μία γραμμή αλλάζει, χωρίς κείμενο αστέρα  (σωστό)')
print()
print('η νέα γραμμή:')
print(' ', diff[0].strip()[:200])

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HUGGINGFACE_API_KEY'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF token φορτώθηκε')
except Exception as e:
    print('Χωρίς HF_TOKEN (εντάξει μόνο για Qwen):', e)


## Ποιο βήμα

In [ ]:
STEP = 1          # <-- ΜΟΝΟ ΑΥΤΟ ΑΛΛΑΖΕΙΣ (1-3)

G2B, Q25 = 'google/gemma-2-2b-it', 'Qwen/Qwen2.5-7B-Instruct'
LLA, Q3, G9B = 'meta-llama/Llama-3.1-8B-Instruct', 'Qwen/Qwen3-4B', 'google/gemma-2-9b-it'
STEPS = {1: [G2B, Q25], 2: [LLA, Q3], 3: [G9B]}
MODELS = STEPS[STEP]
NEEDS_HF = any(m.startswith(('meta-llama/', 'google/')) for m in MODELS)

# (session, σενάριο, σκέλη, πόσα runs περιμένουμε)
#   B: το frame μέσα στα μηνύματα -- μόνο cheap_talk υπάρχει
#   C: το frame στο system prompt -- μόνο το ανοιχτό σκέλος είχε το λάθος
#      prompt· το no_comm δεν έχει καθόλου παράγραφο μηνυμάτων
CALLS_PER_MODEL = [('B', 'framing_competitive', None, 5),
                   ('C', 'framing_competitive_context', ['cheap_talk'], 5)]

def args_for(model, session, scenario, conditions):
    args = ['--model', model, '--session', session, '--topology', 'cycle',
            '--scenarios', scenario, '--games', 'pd',
            '--topology-aware-comm-prompt']
    if conditions:
        args += ['--conditions', *conditions]
    return args

CALLS = [(f'{m.split("/")[-1]} · session {s} · {sc}', args_for(m, s, sc, co), n)
         for m in MODELS for s, sc, co, n in CALLS_PER_MODEL]

print(f'βήμα {STEP}: {len(CALLS)} κλήσεις')
for name, _, n in CALLS:
    print(f'   {name}  -> {n} runs')

## Preview — κοστίζει μηδέν

Σε κάθε κλήση: `topology : cycle`, η γραμμή `PROMPT` με το διορθωμένο prompt,
και το σωστό `expecting`.

In [ ]:
import os, subprocess, sys

ok = True
if NEEDS_HF and not os.environ.get('HUGGINGFACE_API_KEY'):
    ok = False
    print('ΛΕΙΠΕΙ ΤΟ HF_TOKEN -- το μοντέλο είναι gated, θα αποτύχει σε ~1 λεπτό.')
for name, extra, n in CALLS:
    print('=' * 70)
    r = subprocess.run([sys.executable, 'campaign.py', *extra, '--dry-run'],
                       capture_output=True, text=True)
    out = r.stdout.strip()
    print(out[:900])
    if r.returncode != 0:
        ok = False
        print('ΣΦΑΛΜΑ:', (r.stderr or r.stdout).strip()[-300:])
    for needle in ('topology     : cycle', 'topology-aware',
                   f'expecting    : {n} run files'):
        if needle not in out:
            ok = False
            print(f'ΛΕΙΠΕΙ ΑΠΟ ΤΟ ΠΛΑΝΟ: {needle!r}')

assert ok, 'Κάποιο plan απέτυχε -- μη συνεχίσεις.'
print('\n' + '=' * 70)
print('όλα τα πλάνα εντάξει')

## Εκτέλεση

In [ ]:
import subprocess, sys, time

t0 = time.time()
results = []
for name, extra, n in CALLS:
    print('\n' + '=' * 70)
    print(f'βήμα {STEP} · {name}   ({(time.time() - t0) / 3600:.1f} h μέχρι τώρα)')
    print('=' * 70, flush=True)
    s = time.time()
    r = subprocess.run([sys.executable, 'campaign.py', *extra])
    mins = (time.time() - s) / 60
    status = 'OK' if r.returncode == 0 else f'FAILED (exit {r.returncode})'
    results.append((name, status, mins))
    print(f'\n--> {name}: {status}, {mins:.0f} λεπτά', flush=True)

print('\n' + '=' * 70)
print('ΑΠΟΛΟΓΙΣΜΟΣ')
for name, s, mins in results:
    print(f'  {s:22s} {mins:6.0f} λ   {name}')
print(f'\nσύνολο {(time.time() - t0) / 3600:.1f} h')

## Έλεγχος των δεδομένων και πρώτη ματιά

Επιβεβαιώνει δακτύλιο και διορθωμένο prompt, και βάζει δίπλα δίπλα το νέο
νούμερο με το παλιό του πλέγματος. Η γραμμή που έχει σημασία είναι η διαφορά.

In [ ]:
import glob, json, os, sys, collections, statistics
sys.path.insert(0, '.')
from analysis import summarise_run, scenario_of

# Οι τιμές του πλέγματος, με το ΠΑΛΙΟ prompt (δακτύλιος, PD).
OLD = {
    ('gemma-2-2b-it', 'framing_competitive'): 0.034,
    ('gemma-2-2b-it', 'framing_competitive_context'): 0.794,
    ('Qwen2.5-7B-Instruct', 'framing_competitive'): 0.119,
    ('Qwen2.5-7B-Instruct', 'framing_competitive_context'): 0.944,
    ('Llama-3.1-8B-Instruct', 'framing_competitive'): 0.319,
    ('Llama-3.1-8B-Instruct', 'framing_competitive_context'): 0.981,
    ('Qwen3-4B', 'framing_competitive'): 0.019,
    ('Qwen3-4B', 'framing_competitive_context'): 1.000,
    ('gemma-2-9b-it', 'framing_competitive'): 0.006,
    ('gemma-2-9b-it', 'framing_competitive_context'): 1.000,
}

rows = collections.defaultdict(list)
for d in sorted(glob.glob('/kaggle/working/results/*_cycle_commfix*')):
    for p in glob.glob(f'{d}/**/*.json', recursive=True):
        if os.path.basename(p).startswith('_') or f'{os.sep}zips{os.sep}' in p:
            continue
        rec = json.load(open(p, encoding='utf-8'))
        cfg = rec['config']
        sc = cfg.get('scenario', '')
        if not sc.startswith('framing_competitive'):
            continue
        assert rec['topology']['type'] == 'cycle', p
        assert cfg.get('topology_aware_comm_prompt') is True, f'ΛΑΘΟΣ PROMPT: {p}'
        if cfg['condition'] != 'cheap_talk':
            continue
        model = cfg['model']['model_id'].split('/')[-1]
        rows[(model, sc)].append(summarise_run(rec)['coop_rate_overall'])

print(f"{'model':24s} {'cell':32s} {'n':>3s} {'νέο':>7s} {'παλιό':>8s} {'διαφορά':>9s}")
for k in sorted(rows):
    v = rows[k]
    new = statistics.fmean(v)
    old = OLD.get(k, float('nan'))
    print(f'{k[0]:24s} {k[1]:32s} {len(v):3d} {new:7.3f} {old:8.3f} {new - old:+9.3f}')
print('\nΤο ablation του session A είχε δώσει μετακινήσεις εντός θορύβου· '
      'ό,τι εδώ ξεπερνά ±0,15 αξίζει δεύτερη ματιά.')

## Μάζεμα

In [ ]:
import glob, os, shutil

zips = sorted(glob.glob('/kaggle/working/*_cycle_session[BC]_commfix*.zip'))
print(f'{len(zips)} zip:')
for z in zips:
    print(f'   {os.path.getsize(z)/1e6:6.1f} MB  {os.path.basename(z)}')
if zips:
    box = '/kaggle/working/ring_competitive'
    os.makedirs(box, exist_ok=True)
    for z in zips:
        shutil.copy(z, box)
    tag = '_'.join(m.split('/')[-1] for m in MODELS)
    out = shutil.make_archive(f'/kaggle/working/ring_competitive_step{STEP}_{tag}',
                              'zip', box)
    print(f'\nκατέβασε αυτό: {out}  ({os.path.getsize(out)/1e6:.1f} MB)')
else:
    print('\nΚΑΝΕΝΑ ZIP -- δες τον απολογισμό παραπάνω.')

## Μετά

Αποσυμπίεσε στη ρίζα του `diplomatikh`. Τα runs του session B μπαίνουν στον
ίδιο φάκελο με το ablation, `<model>_cycle_commfix`, και του session C σε
`<model>_cycle_commfix_cheaptalk`, γιατί έτρεξε μόνο το ανοιχτό σκέλος.

**Πώς διαβάζεται.** Δύο συγκρίσεις ανά μοντέλο, νέο απέναντι σε παλιό:

1. `framing_competitive`: αν μείνει στο ίδιο επίπεδο, η μοναδική συστηματική
   οδός βλάβης μετρήθηκε σωστά παρά το λάθος prompt.
2. `framing_competitive_context[cheap_talk]`: αν μείνει στο 0,79–1,00, ο
   ισχυρισμός «ανοιχτό κανάλι υπερισχύει του frame στο prompt» στέκει και
   στον δακτύλιο με σωστή περιγραφή δικτύου.

Αν και τα δέκα κελιά μείνουν εντός θορύβου, το κείμενο μπορεί να πει ότι το
bug ελέγχθηκε **στα κελιά που κουβαλούν κάθε ισχυρισμό**, και όχι μόνο στο
session A.